# Inverse Problems 2026 - Exercise 1 (home exercises)

Return by **Fri 6.11.2026 at 23:59** in Moodle! You can ask for help with these exercises in the exercise session the same day (10:00, room 7334).

The returned exercises should be renamed `Exercise01_lastname_firstname.ipynb` and contain your solutions in clear and readable manner and format. The programming exercises need to be runnable with minimal effort. If you find markdown too cumbersome for writing the necessary math, you can attach **a single extra pdf-file** for the theoretical exercises and denote it `Exercise01_lastname_firstname.pdf`.

___
## Task 1
The following Python code reproduces the results for the inverse heat distribution problem obtained in the first set of lecture notes. Modify it for the following setup.

Use the end time $T = 0.05$. Whatâ€™s the rank of $A$ now? Add similar noise to the observation with standard deviation $0.001$
as in the slides. How many singular values can be used before reconstruction becomes unstable? Include a plot of your solution in the answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

T = 0.1
N = 100
h = np.pi/N
B = np.diag(np.ones(N-2), k=-1) - 2*np.eye(N-1) + np.diag(np.ones(N-2), k=1) # Tridiagonal matrix
B[0,0] = -1 # left boundary condition
B[-1,-1] = -1 # right boundary condition
B = B / h**2 # B corresponds to second derivative

A = la.expm(T * B) # Matrix exponential
[U, Svals, Vt] = la.svd(A) # Singular Value Decomposition

plt.semilogy(Svals)
plt.title("Singular values of A");

In [ ]:
x = np.linspace(h, np.pi, N-1).T
a, b1, b2 = 40*np.pi/30, -8/3, 20/3
peak = 35
f = np.maximum(np.concatenate([a * x[:peak] + b1, -a* x[peak:] + b2]), 0) # Ground truth solution
w = A @ f # Data

out = la.lstsq(A, w)
f_stupid = out[0]

fig = plt.figure(figsize=(12,5))
ax1 = fig.add_subplot(1,2,1)
ax1.plot(x, f, 'k', x, w, 'r')

ax2 = fig.add_subplot(1,2,2)
ax2.plot(x, f_stupid);

___
In the following exercises we learn to solve 1D deconvolution problems. The direct problem is defined as

$$ g(x) = Af(x) = \int k(x âˆ’y)f(y)dy, $$

where the domain of integration is a unit interval. As you might remember from Functional Analysis the mapping $A : C[0,1) \to C[0,1)$ is compact and, therefore, the inverse problem is ill-posed.
___

## Task 2
Run the following code cell. Plot the reconstructed vectors `recn1`, `recn2` and `recn3`, then explain how each is obtained. How does an _inverse crime_ affect the reconstructions?

In [ ]:
rng = np.random.default_rng(42)

n = 64
xx = np.linspace(0, 1, n)

def fun(x):
    return 0.05*((4*x)**2 - 4) + 0.05*np.mod(14*x,2.5)**4

psf = np.array([.05, .2, .45, .25, .05]) # Point spread function

A = la.convolution_matrix(psf, n, mode="same")
B = la.inv(A)

data_clean = A @ fun(xx)

sigma = 0.1 # Noise level
data_noisy = data_clean + sigma*rng.standard_normal(n)

recn1 = B @ data_clean
recn2 = B @ data_noisy
recn3 = fun(xx)

___
## Task 3 (Truncated SVD)
Apply truncated SVD with the following code. The routine plots TSVD solution with truncation levels corresponding to saving 5, 10, 20, 40, 60, 80 and 100% of the singular values. Hack this routine and figure what is the optimal truncation level for your unknown assuming you don't know the function `fun`? What criterion would you use? (There are multiple ways of optimizing and no single right answers.)

As default, the routine adds normal random noise to the discrete data with standard deviation $\sigma = 0.1$. Produce data also with $\sigma = 0.02$ and $\sigma = 0.2$. How do these choices affect the 'optimal' truncation level?

In [ ]:
data_clean = np.convolve(fun(np.linspace(0, 1, 2*n)), psf, mode="same")[::2] # Data simulated at twice the resolution
ground_truth = fun(xx)

sigma = 0.1 # Noise level
data_noisy = data_clean + sigma*rng.standard_normal((n))

[U, Svals, Vt] = la.svd(A) # Singular Value Decomposition

sv_percents = np.array([5, 10, 20, 40, 60, 80, 100])

fig, axs = plt.subplots(len(sv_percents) + 2, sharex=True)
fig.set_size_inches((5, 10))
fig.suptitle("Truncated SVD reconstructions")

axs[0].plot(xx, ground_truth, 'k')
axs[0].set_title("Ground truth")

axs[1].plot(xx, data_noisy, 'r')
axs[1].set_title("Noisy data")

for iii, p in enumerate(sv_percents[::-1]):
    nsv = np.int16(n * p / 100) # Number of singular values used
    Sp_inv = np.zeros(n)
    Sp_inv[:nsv] = 1 / Svals[:nsv]
    Sp_inv = np.diag(Sp_inv)

    recn = Vt.T @ Sp_inv @ U.T @ data_noisy

    axs[iii+2].plot(xx, recn)
    axs[iii+2].set_title(f"Using {nsv} ({p}%) singular values")
fig.tight_layout()

___
## Task 4 (Convolution and its adjoint)

Given two (sufficiently well-behaving) functions $f,g: \mathbb{R}^d \to \mathbb{R}$, their convolution is defined as the linear integral operator
$$ f*g(x) := \int_{\mathbb{R}^d} f(y) g(x - y) dy.$$
Show that
1. $f * g = g * f$,
2. $f * (g * h) = (f * g) * h$,
3. $f * (g + h) = f * g + f * h$,
4. $\widehat{f * g} = \widehat{f} \, \widehat{g}$, where $\widehat{f}(\xi) = \int_{\mathbb{R}^d} f(x) e^{-2 \pi i x \cdot \xi} dx$ is the Fourier transform.

For a fixed sequence $g \in \ell^2(\mathbb{Z})$ (sometimes called the _convolution filter_ or _convolution kernel_), consider the operator $G: \ell^2(\mathbb{Z}) \to \ell^2(\mathbb{Z})$ defined as the following _discrete convolution_
$$ G(f) = f * g = \left( \sum_{i = -\infty}^\infty f_i g_{j - i} \right)_{j=-\infty}^\infty. $$
5. Find its adjoint $G^*$ and in particular, what is the filter of $G^*$?
